# Stress test — SIMPLACE against torchcrop, same crop, same day, no spin-up

The smoke test asks *how close to observations* each model gets and accepts
that they differ in every respect. This asks the narrower question that one
cannot answer: **with every difference that is not the model itself removed, do
the two agree?** The run is produced by
[`submit/submit_stresstest.py`](../submit/submit_stresstest.py), which removes
them one at a time:

| Confound | How it is removed |
|---|---|
| Crop parameters | torchcrop is loaded from SIMPLACE's own `crop.xml` |
| Sowing date | SIMPLACE runs first; torchcrop is latched to its **simulated** `PlantingDOY` |
| Spin-up | Both start at sowing, from the export's initial soil water |
| Irrigation | Removed from both — the solution has no irrigation module |
| CO₂ | Held at 360 ppm, where the crop file's own response curve is 1.0 |
| Fertilizer | Not adjusted — **checked**, in §3 |

**There is no observation in this notebook.** Neither model is a reference, so
every number below is a difference between two simulations of the same site and
season, signed `torchcrop − simplace`. A disagreement says the two models are
not the same model; it does not say which is right. For that, read
[`germany_smoke_evaluation.ipynb`](germany_smoke_evaluation.ipynb).

```bash
./submit/submit_stresstest.py                 # both scenarios, both seasons
./submit/submit_stresstest.py --dry-run       # audit the parameters, run nothing
```

## 0. Setup

In [ ]:
import logging
import sys
from pathlib import Path

# cropmodelling4eu is installed (pip install -e .), so the evaluation
# library is imported like any other package rather than off sys.path.

import numpy as np
import pandas as pd

from cropmodelling4eu.evaluation import aggregate, config, cybench, doy, metrics, plots, regions, torchcrop
from cropmodelling4eu.evaluation.style import use_style

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s",
                    force=True)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

PALETTE = use_style("light")
config.ensure_output_dirs()

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 140)

print(f"TorchCrop run : {config.TORCHCROP_RUN_DIR}")
print(f"CyBench root  : {config.CYBENCH_ROOT}")
print(f"Outputs       : {config.OUTPUT_DIR}")

In [ ]:
from cropmodelling4eu.evaluation import stresstest as st

ROOT = st.DEFAULT_ROOT

run = st.load_run(ROOT)
paired = st.pair(run)
provenance = st.load_provenance(ROOT)

print(f"root      : {ROOT}")
print(f"cells     : {run['SimplaceID'].nunique()}")
print(f"seasons   : {sorted(run['year'].unique())}")
print(f"scenarios : {sorted(run['scenario'].unique())}")
print(f"paired    : {len(paired)} cell-seasons both models ran")

## 1. What this run actually removed

Read from `torchcrop/config.yaml`, which is not a report: the script writes it,
loads it back, and the `RunConfig` in it is what both halves were driven by. So
this is the run's input, and editing it and re-running with `--reuse-simplace`
re-runs the torchcrop half exactly as edited.

In [ ]:
print(provenance.get("purpose", "no provenance block found"), "\n")
for name, note in provenance.get("removed_inputs", {}).items():
    print(f"  {name:12s} {note}")
print()
for name, scenario in provenance.get("scenarios", {}).items():
    print(f"  {name:12s} iopt={scenario['iopt']}  {scenario['note']}")

### The asymmetries the design could **not** remove

These are recorded in the config rather than papered over, and both bound how
far §4 can be read.

In [ ]:
for note in provenance.get("known_asymmetries", []):
    print("* " + note + "\n")

## 2. Crop parameters — are the two models growing the same crop?

The audit runs before anything else. With `--crop-params simplace` (the
default) torchcrop is *built from* `crop.xml`, so the remaining differences are
parameters SIMPLACE has no counterpart for, or a mapping that could not be
made. Run with the shipped presets instead and 21 of 72 differ — including
`TSUM1` (1623 vs 1050) — and no disagreement below is attributable to the model.

In [ ]:
audit = st.load_audit(ROOT)
print(audit["status"].value_counts().to_string())
print(f"\ntorchcrop crop parameters: {provenance.get('crop_parameters', {}).get('torchcrop_source')}")
display(audit[audit["status"] != "same"][["parameter", "kind", "simplace", "torchcrop", "status"]])

## 3. Two checks that must pass before anything else is read

**The sowing latch.** torchcrop is set to SIMPLACE's simulated `PlantingDOY`
per cell and season. Any row here means it fell back to the export's calendar
instead, and those cells are then two different seasons rather than two models.

**The fertilizer.** Both read one schedule — the export's
`fertilizer_<crop>.csv` — but by different routes: SIMPLACE takes product
amounts and carrier contents from `fertilizer_composition.xml`, torchcrop
converts them to nutrient rates before the run. Two conversions of one file is
exactly where a silent factor hides.

In [ ]:
mismatched = st.check_latches(paired)
fertilizer = st.load_fertilizer_check(ROOT)

print(f"sowing latch : {len(mismatched)} of {len(paired)} cell-seasons differ  "
      f"({'PASS' if mismatched.empty else 'FAIL'})")
print(f"fertilizer N : largest |difference| {fertilizer['difference'].abs().max():.2e} g N/m², "
      f"mean applied {fertilizer['n_applied_g_m2_simplace'].mean():.2f} g N/m²  "
      f"({'PASS' if fertilizer['difference'].abs().max() < 1e-3 else 'FAIL'})")
display(mismatched.head())

## 4. Agreement, quantity by quantity

Ordered from the result to the diagnostics that explain it. `bias` and `rmse`
are in the quantity's own unit; `ratio` is `torchcrop / simplace` on the means,
which is what travels between a 4 t/ha cell and a 9 t/ha one.

`r` is not a skill score here. Both sides vary across cells and seasons for
their own reasons, so a low `r` with a small bias means the two models disagree
about *which* cells are good ones — often a more serious finding than a level
offset, and invisible in the mean.

In [ ]:
scores = st.agreement(paired)
display(scores.round(3))
scores.to_csv(config.TABLE_DIR / "stresstest_agreement.csv", index=False,
              float_format="%.4f")

for variable in st.COMPARED:
    print(f"  {variable.label:14s} {variable.note}")

In [ ]:
fig, axes = st.plot_agreement(paired, palette=PALETTE)
fig.savefig(config.FIGURE_DIR / "stresstest_agreement.png", dpi=150, bbox_inches="tight")

## 5. Which cells disagree

A scatter hides *which* cells the disagreement sits in; this does not. Read the
shape, not the individual rules: a long rule on one cell with none on its
neighbours is a site problem (a soil, a failed establishment), while a fan that
widens with the SIMPLACE value is a systematic difference in the response.

In [ ]:
fig, axes = st.plot_cell_pairs(paired, "yield_t_ha", palette=PALETTE)
fig.savefig(config.FIGURE_DIR / "stresstest_cell_yields.png", dpi=150, bbox_inches="tight")

In [ ]:
worst = (
    paired.assign(ratio=paired["yield_ratio"])
    .sort_values("ratio")
    [["scenario", "SimplaceID", "year", "lon", "lat", "yield_t_ha_simplace",
      "yield_t_ha_torchcrop", "ratio", "max_lai_simplace", "max_lai_torchcrop",
      "tranrf_mean_simplace", "tranrf_mean_torchcrop"]]
)
print("The ten cell-seasons where torchcrop is furthest below SIMPLACE:")
display(worst.head(10).round(2))
paired.to_csv(config.TABLE_DIR / "stresstest_paired_cells.csv", index=False,
              float_format="%.4f")

## 6. Where the divergence comes from

The yield ratio against the difference in each state variable. Which one it
tracks is the answer the tables above cannot give:

* it falls with **Δ peak LAI** → a canopy that never built, so the difference is
  in growth or establishment;
* it falls with **Δ TRANRF** → the water balances closed the stomata at
  different times, which is the bucket-vs-layered asymmetry the config records;
* it falls with **Δ season length** → the phenology integrated different
  temperatures despite the shared `TSUM1`;
* it tracks **nothing** → partitioning, and the season means do not resolve it —
  go to the daily trajectories in §7.

In [ ]:
fig, axes = st.plot_divergence(paired, palette=PALETTE)
fig.savefig(config.FIGURE_DIR / "stresstest_divergence.png", dpi=150, bbox_inches="tight")

## 7. Inside the season

A season mean cannot say *when* the two models parted, and that is the whole
question: `TRANRF` averaging 0.8 is a season mildly stressed throughout and a
season shut down for three weeks in June, and those are different crops.

Unlike the smoke test's version of this figure, `das` here is the **same
calendar day in both models** — that is what the sowing latch of §3 buys. A
horizontal offset between the curves is therefore a difference in development
rate, not in the day the crop went in. The band is the interquartile range
**across cells**, not an uncertainty; one model's band being far wider than the
other's is itself a result.

In [ ]:
SCENARIO = "limited"
daily = st.load_daily(ROOT, scenarios=[SCENARIO])
season = (
    daily.groupby(["variable", "year", "model"])["value"]
    .agg(mean="mean", peak="max", trough="min")
    .round(3)
    .unstack("model")
)
display(season)
season.to_csv(config.TABLE_DIR / "stresstest_daily_season.csv")

In [ ]:
fig, axes = st.plot_daily(daily, SCENARIO, palette=PALETTE)
fig.savefig(config.FIGURE_DIR / "stresstest_daily.png", dpi=150, bbox_inches="tight")

## 8. The nutrient-limitation response

`potential` (IOPT=1) minus `limited` (IOPT=3), taken **within** each model. The
question is not whether the two agree in level — §4 answered that — but whether
they respond to `iopt` by a similar amount.

Two things to keep in mind. IOPT=1 is not potential production in either model:
both apply water stress to growth unconditionally, so this is a
nutrient-unlimited but still water-limited run on both sides. And a model that
ran only one scenario is dropped from the figure with a warning rather than
plotted against its own missing half.

In [ ]:
effect = st.scenario_effect(run, "yield_t_ha")
if effect.empty:
    print("only one scenario is present in this run — nothing to compare")
else:
    display(
        effect.groupby("model")[["effect", "effect_percent"]]
        .describe().round(2).T
    )
    fig, axes = st.plot_scenario_effect(effect, palette=PALETTE)
    fig.savefig(config.FIGURE_DIR / "stresstest_scenario_effect.png", dpi=150,
                bbox_inches="tight")

## 9. Summary

In [ ]:
overview = st.summary(paired)
overview.to_frame().to_csv(config.TABLE_DIR / "stresstest_summary.csv")
print(f"tables written to {config.TABLE_DIR}")
print(f"figures written to {config.FIGURE_DIR}")
overview.to_frame()

### Reading it

* **The checks in §3 are pass/fail, not diagnostics.** A sowing latch mismatch
  or a fertilizer difference above ~1e-3 g N/m² invalidates everything after
  it; fix the run before reading §4.
* **A level offset and a rank disagreement are different findings.** A ratio of
  0.8 with `r ≈ 0.9` is one model consistently lower — a calibration
  difference. A ratio near 1 with `r ≈ 0.1` is the two models disagreeing about
  which cells are good, which no bias correction repairs.
* **Follow the chain, not the yield.** Yield is the last quantity in it: peak
  LAI explains biomass, biomass and partitioning explain yield, and §6 says
  which link the run broke.
* **Cells where torchcrop collapses to a fraction of SIMPLACE** (the last row of
  the summary) are worth reading individually before any pooled statistic —
  a handful of failed establishments moves a mean far more than a systematic
  offset does.
* **This notebook cannot say which model is right.** It says whether the two are
  the same model given the same inputs, and where they stop being one.